<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/llm/SmolLM/smollm-q12-Question.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("smollm", ...)`


# Implement SmolLM-135M from Scratch

### Problem Statement
[SmolLM-135M](https://huggingface.co/HuggingFaceTB/SmolLM-135M) is a small decoder-only language model with the Llama architecture. Build it from scratch in PyTorch, so that the pretrained weights load into your modules and your logits match the reference `transformers` model.

### Architecture (from the checkpoint's `config.json`)

| | |
|---|---|
| vocabulary | 49,152 tokens |
| hidden size | 576 |
| decoder layers | 30 |
| attention | 9 query heads, 3 key/value heads (grouped-query attention), head_dim 64, no biases |
| positions | RoPE, base θ = 10000, `rotate_half` (half-split) pairing |
| MLP | SwiGLU: `W_down(silu(W_gate(x)) * W_up(x))`, intermediate size 1536 |
| normalization | RMSNorm, eps 1e-5, **pre-norm** (before attention and before the MLP), plus a final norm |
| output | `lm_head` tied to the input embedding (`tie_word_embeddings`) |

That is about 134.5M parameters (the tied embedding counted once). SmolLM**2**-135M has the same shapes but RoPE base 100000.

### Requirements
1. **Attention block** (graded with `check("smollm", ...)`): `rotate_half`, `apply_rotary_pos_emb`, `repeat_kv` (KV head `i // n_rep` serves query head `i`), `RotaryEmbedder` and `RopeAttention`. `RopeAttention.forward(hidden_states, attention_mask)` takes an **additive** float mask broadcastable to `(batch, heads, seq, seq)`: 0 where attending is allowed, a very negative number where it is not.
2. **The rest of the model**: `MLP`, `RMSNorm`, `LlamaDecoder` (residual pre-norm block), `smolModel` (embeddings → layers → final norm; it builds the causal + padding mask once from the `(batch, seq)` attention mask) and `smolLM` (adds the tied `lm_head` and returns `{"logits": ...}`).
3. Use the module names below, so that `load_hf_weights` can copy the pretrained weights in.
4. **Test**: a quick CPU test with a tiny config, then load the real weights and check that your logits match `transformers`' to ~1e-4 and that greedy decoding produces the same text.

```
smolLM(
  (model): smolModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoder(
        (self_attn): RopeAttention(
          (W_query): Linear(in_features=576, out_features=576, bias=False)
          (W_key): Linear(in_features=576, out_features=192, bias=False)
          (W_value): Linear(in_features=576, out_features=192, bias=False)
          (W_output): Linear(in_features=576, out_features=576, bias=False)
          (rotary_emb): RotaryEmbedder()
        )
        (mlp): MLP(
          (W_gate): Linear(in_features=576, out_features=1536, bias=False)
          (W_up): Linear(in_features=576, out_features=1536, bias=False)
          (W_down): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLU()
        )
        (pre_attn_rmsnorm): RMSNorm()
        (pre_mlp_rmsnorm): RMSNorm()
      )
    )
    (norm): RMSNorm()
  )
  (lm_head): Linear(in_features=576, out_features=49152, bias=False)
)
```

In [ ]:
import math
import torch
from torch import nn
import torch.nn.functional as F


def rotate_half(x):
    """(..., d) -> (..., d): split the last dim into halves (x1, x2), return cat(-x2, x1)."""
    ...


def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    """q: (batch, num_heads, seq, head_dim), k: (batch, kv_heads, seq, head_dim),
    cos/sin: (batch or 1, seq, head_dim). Unsqueeze cos/sin at `unsqueeze_dim` so they
    broadcast over the heads, then rotate q and k. Returns (q_rot, k_rot)."""
    ...


def repeat_kv(hidden_states, n_rep):
    """(batch, kv_heads, seq, head_dim) -> (batch, kv_heads * n_rep, seq, head_dim),
    where output head i is a copy of KV head i // n_rep."""
    ...


class RotaryEmbedder(nn.Module):
    def __init__(self, dim, base):
        super().__init__()
        # TODO: inverse frequencies base^(-2i/dim), i = 0 .. dim/2 - 1, as a buffer
        ...

    @torch.no_grad()
    def forward(self, x):
        """x: (..., seq, head_dim). Return (cos, sin), each (1, seq, head_dim), for
        positions 0 .. seq-1, with the angles laid out as cat(angles, angles)."""
        ...


class RopeAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        # TODO: W_query, W_key, W_value, W_output (no bias; W_key/W_value project to
        # kv_heads * head_dim) and self.rotary_emb = RotaryEmbedder(dim=head_dim, base=config.rope_theta)
        ...

    def forward(self, hidden_states, attention_mask=None):
        """hidden_states: (batch, seq, hidden_size); attention_mask: additive float mask
        broadcastable to (batch, num_heads, seq, seq). Returns (batch, seq, hidden_size)."""
        # TODO: project, split heads, rotate q and k, repeat_kv, scaled dot-product
        # attention + mask, softmax, merge heads, W_output
        ...

In [ ]:
class MLP(nn.Module):
    """SwiGLU: W_down(silu(W_gate(x)) * W_up(x)), no biases."""
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        ...

    def forward(self, x):
        ...


class RMSNorm(nn.Module):
    """x / sqrt(mean(x^2, dim=-1) + eps) * weight, with a learnable `weight` of ones."""
    def __init__(self, hidden_size, eps=1e-5):
        super().__init__()
        ...

    def forward(self, hidden_states):
        ...


class LlamaDecoder(nn.Module):
    """Pre-norm block: x = x + self_attn(pre_attn_rmsnorm(x), mask); x = x + mlp(pre_mlp_rmsnorm(x))."""
    def __init__(self, config):
        super().__init__()
        ...

    def forward(self, hidden_states, attention_mask=None):
        ...


class smolModel(nn.Module):
    """embed_tokens -> layers -> norm. Builds the additive causal (+ padding) mask once."""
    def __init__(self, config):
        super().__init__()
        ...

    def forward(self, input_ids, attention_mask=None):
        # attention_mask: (batch, seq), 1 = real token, 0 = padding
        ...


class smolLM(nn.Module):
    """smolModel + lm_head tied to model.embed_tokens. forward returns {"logits": logits}."""
    def __init__(self, config):
        super().__init__()
        ...

    def forward(self, input_ids, attention_mask=None):
        ...


class smolConfig:
    # SmolLM-135M (HuggingFaceTB/SmolLM-135M config.json)
    vocab_size = 49152
    hidden_size = 576
    intermediate_size = 1536
    num_hidden_layers = 30
    num_heads = 9          # num_attention_heads -> head_dim = 64
    kv_heads = 3           # num_key_value_heads (GQA, 3 query heads per K/V head)
    rope_theta = 10000.0
    rms_norm_eps = 1e-5

## Test

In [ ]:
# Quick CPU checks with a tiny config (no download needed)
class TinyConfig(smolConfig):
    vocab_size, hidden_size, intermediate_size = 100, 48, 96
    num_hidden_layers, num_heads, kv_heads = 2, 6, 2

torch.manual_seed(0)
tiny = smolLM(TinyConfig()).eval()
ids = torch.randint(0, 100, (2, 10))
with torch.no_grad():
    logits = tiny(ids)["logits"]
    assert logits.shape == (2, 10, 100)

    # Causal: changing the last token must not change any earlier logits.
    ids2 = ids.clone()
    ids2[:, -1] = (ids2[:, -1] + 1) % 100
    assert torch.allclose(tiny(ids2)["logits"][:, :-1], logits[:, :-1], atol=1e-5)

    # Padding: left-pad row 0 with 3 masked tokens. Its real tokens now sit at
    # positions 3..12, but RoPE only sees relative offsets and the padding is
    # masked out, so their logits must not change.
    padded = torch.cat([torch.zeros(1, 3, dtype=torch.long), ids[:1]], dim=1)
    mask = torch.cat([torch.zeros(1, 3), torch.ones(1, 10)], dim=1).long()
    assert torch.allclose(tiny(padded, mask)["logits"][:, 3:], logits[:1], atol=1e-4)
assert tiny.lm_head.weight is tiny.model.embed_tokens.weight, "lm_head must be tied"

n_params = sum(p.numel() for p in smolLM(smolConfig()).parameters())
print(f"SmolLM-135M has {n_params / 1e6:.1f}M parameters")  # 134.5M
print("tiny model: shapes, causal mask, padding mask and tied weights OK")

In [ ]:
# Load the pretrained SmolLM-135M weights into our model and compare with transformers.
# Needs `pip install transformers` and a ~270 MB download.
from transformers import AutoTokenizer, AutoModelForCausalLM

# Hugging Face Llama parameter names -> the names used in this notebook
RENAME = {
    "self_attn.q_proj": "self_attn.W_query", "self_attn.k_proj": "self_attn.W_key",
    "self_attn.v_proj": "self_attn.W_value", "self_attn.o_proj": "self_attn.W_output",
    "mlp.gate_proj": "mlp.W_gate", "mlp.up_proj": "mlp.W_up", "mlp.down_proj": "mlp.W_down",
    "input_layernorm": "pre_attn_rmsnorm", "post_attention_layernorm": "pre_mlp_rmsnorm",
}

def load_hf_weights(model, hf_model):
    state = {}
    for name, tensor in hf_model.state_dict().items():
        for old, new in RENAME.items():
            name = name.replace(old, new)
        state[name] = tensor
    state.setdefault("lm_head.weight", state["model.embed_tokens.weight"])  # tied
    model.load_state_dict(state, strict=True)  # every weight must find its slot

checkpoint = "HuggingFaceTB/SmolLM-135M"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
reference_model = AutoModelForCausalLM.from_pretrained(checkpoint).float().eval()

test_model = smolLM(smolConfig()).eval()
load_hf_weights(test_model, reference_model)

prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    ours = test_model(**inputs)["logits"]
    ref = reference_model(**inputs).logits
print(f"max |logit difference|: {(ours - ref).abs().max().item():.2e}")
assert torch.allclose(ours, ref, atol=1e-3), "logits do not match the reference model"


@torch.no_grad()
def greedy_generate(model, input_ids, max_new_tokens=10):
    for _ in range(max_new_tokens):
        next_id = model(input_ids)["logits"][:, -1].argmax(dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_id], dim=-1)
        if next_id.item() == tokenizer.eos_token_id:
            break
    return input_ids

ours_text = tokenizer.decode(greedy_generate(test_model, inputs["input_ids"])[0])
ref_text = tokenizer.decode(greedy_generate(lambda ids: {"logits": reference_model(ids).logits},
                                            inputs["input_ids"])[0])
print("ours:     ", repr(ours_text))
print("reference:", repr(ref_text))
assert ours_text == ref_text